# Geo Poker Mobile / PWA Version

Dieses Notebook ist unabhängig vom bisherigen Browser-Prototyp.

Ziel:
- Einen neuen Ordner `GeoPokerMobile/` erzeugen
- Mobile-freundliche Dateien schreiben
- Die vorhandenen CSV-Listen aus `Uppsala/Lists/` verwenden
- `places_data.js` automatisch erzeugen
- `country_outlines.js` automatisch erzeugen
- Eine installierbare Progressive Web App vorbereiten

Dein bisheriges `Uppsala.ipynb` bleibt unverändert.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import urllib.request
import shutil

BASE = Path("/Users/kathsch/Desktop/Uppsala")
LISTS = BASE / "Lists"

APP = BASE / "GeoPokerMobile"
APP.mkdir(exist_ok=True)

print("Mobile-App-Ordner:")
print(APP)
print()
print("Listen-Ordner existiert:", LISTS.exists())


## 1. `places_data.js` aus allen CSV-Dateien erzeugen

Alle Dateien nach dem Muster `*_orte_koordinaten.csv` im Ordner `Lists/` werden automatisch eingebunden.


In [ ]:
js = "const COUNTRY_DATA = {};\n\n"

for csv_path in sorted(LISTS.glob("*_orte_koordinaten.csv")):
    country = csv_path.stem.replace("_orte_koordinaten", "").lower()
    print("Loading:", country)

    df = pd.read_csv(csv_path)

    js += (
        f"COUNTRY_DATA['{country}'] = "
        + json.dumps(df.to_dict("records"), ensure_ascii=False, indent=2)
        + ";\n\n"
    )

(APP / "places_data.js").write_text(js, encoding="utf-8")

print()
print("Created:", APP / "places_data.js")


## 2. `country_outlines.js` erzeugen

Diese Zelle lädt Natural Earth Ländergrenzen und erzeugt Umrisse für Länder und Regionen.

Wenn du später neue Regionen ergänzt, musst du sie nur im Dictionary `wanted` ergänzen.


In [ ]:
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"

with urllib.request.urlopen(url) as response:
    world = json.load(response)

wanted = {
    # countries
    "italy": ["Italy"],
    "germany": ["Germany"],
    "france": ["France"],
    "switzerland": ["Switzerland"],
    "spain": ["Spain"],
    "usa": ["United States of America"],
    "india": ["India"],
    "china": ["China"],
    "russia": ["Russia"],
    "australia": ["Australia"],
    "brazil": ["Brazil"],

    # regions
    "africa": [
        "Morocco", "Algeria", "Tunisia", "Libya", "Egypt",
        "Mauritania", "Mali", "Niger", "Chad", "Sudan",
        "South Sudan", "Eritrea", "Djibouti", "Ethiopia", "Somalia",
        "Senegal", "Gambia", "Guinea-Bissau", "Guinea", "Sierra Leone",
        "Liberia", "Côte d'Ivoire", "Ghana", "Togo", "Benin",
        "Burkina Faso", "Nigeria", "Cameroon", "Central African Republic",
        "Equatorial Guinea", "Gabon", "Republic of the Congo",
        "Democratic Republic of the Congo", "Uganda", "Rwanda", "Burundi",
        "Kenya", "Tanzania", "Angola", "Zambia", "Malawi",
        "Mozambique", "Zimbabwe", "Botswana", "Namibia",
        "South Africa", "Lesotho", "Eswatini", "Madagascar"
    ],
    "middle_america": [
        "Mexico", "Guatemala", "Belize", "Honduras", "El Salvador",
        "Nicaragua", "Costa Rica", "Panama", "Cuba", "Jamaica",
        "Haiti", "Dominican Republic", "Puerto Rico", "Bahamas",
        "Trinidad and Tobago"
    ],
    "south_america": [
        "Colombia", "Venezuela", "Guyana", "Suriname", "Ecuador",
        "Peru", "Brazil", "Bolivia", "Paraguay", "Chile",
        "Argentina", "Uruguay"
    ],
    "europe": [
        "Portugal", "Spain", "France", "Belgium", "Netherlands",
        "Luxembourg", "Germany", "Switzerland", "Austria", "Italy",
        "Ireland", "United Kingdom", "Norway", "Sweden", "Finland",
        "Denmark", "Iceland", "Poland", "Czechia", "Slovakia",
        "Hungary", "Slovenia", "Croatia", "Bosnia and Herzegovina",
        "Serbia", "Montenegro", "Albania", "North Macedonia",
        "Greece", "Bulgaria", "Romania", "Moldova", "Ukraine",
        "Belarus", "Lithuania", "Latvia", "Estonia"
    ],
    "southeast_asia": [
        "Myanmar", "Thailand", "Laos", "Cambodia", "Vietnam",
        "Malaysia", "Singapore", "Indonesia", "Brunei",
        "Philippines", "Timor-Leste"
    ],
    "central_asia": [
        "Kazakhstan", "Uzbekistan", "Turkmenistan", "Kyrgyzstan",
        "Tajikistan", "Afghanistan", "Mongolia"
    ],
    "middle_east": [
        "Turkey", "Cyprus", "Syria", "Lebanon", "Israel",
        "Palestine", "Jordan", "Iraq", "Iran", "Saudi Arabia",
        "Kuwait", "Qatar", "United Arab Emirates", "Oman",
        "Yemen", "Georgia", "Armenia", "Azerbaijan"
    ],
    "oceania": [
        "Australia", "New Zealand", "Papua New Guinea",
        "Fiji", "Solomon Islands", "Vanuatu", "New Caledonia"
    ],
}

def feature_name(feature):
    props = feature["properties"]
    return props.get("NAME") or props.get("ADMIN") or ""

name_to_geometry = {
    feature_name(feature): feature["geometry"]
    for feature in world["features"]
}

outlines = {}
missing_by_key = {}

for key, names in wanted.items():
    geometries = []
    missing = []

    for name in names:
        if name in name_to_geometry:
            geometries.append(name_to_geometry[name])
        else:
            missing.append(name)

    if missing:
        missing_by_key[key] = missing

    if len(geometries) == 1:
        outlines[key] = geometries[0]
    else:
        outlines[key] = {
            "type": "GeometryCollection",
            "geometries": geometries
        }

if missing_by_key:
    print("Nicht gefunden:")
    for key, names in missing_by_key.items():
        print(key, ":", names)

js = "const COUNTRY_OUTLINES = " + json.dumps(outlines, ensure_ascii=False) + ";"

(APP / "country_outlines.js").write_text(js, encoding="utf-8")

print()
print("Created:", APP / "country_outlines.js")
print("Outlines:", len(outlines))


## 3. Mobile `index.html`

Diese Datei enthält PWA-Metadaten und lädt alle App-Dateien.


In [ ]:
html = """<!DOCTYPE html>
<html lang="de">
<head>
  <meta charset="UTF-8">
  <title>Geo Poker</title>
  <meta name="viewport" content="width=device-width, initial-scale=1.0, viewport-fit=cover">

  <link rel="manifest" href="manifest.json">
  <meta name="theme-color" content="#1f2937">
  <meta name="apple-mobile-web-app-capable" content="yes">
  <meta name="apple-mobile-web-app-title" content="Geo Poker">

  <link rel="stylesheet" href="style.css">
</head>

<body>
  <div id="app">
    <header>
      <h1>Geo Poker</h1>

      <div class="controls">
        <label for="countrySelect">Land / Region</label>
        <select id="countrySelect"></select>

        <button onclick="newGame()">Neues Spiel</button>
        <button onclick="stopGame()">Punkte sichern</button>
      </div>

      <div id="status"></div>
      <div id="message"></div>
    </header>

    <main>
      <div id="board"></div>
    </main>
  </div>

  <script src="places_data.js"></script>
  <script src="country_outlines.js"></script>
  <script src="game.js"></script>

  <script>
    if ("serviceWorker" in navigator) {
      navigator.serviceWorker.register("service-worker.js");
    }
  </script>
</body>
</html>
"""

(APP / "index.html").write_text(html, encoding="utf-8")
print("Created:", APP / "index.html")


## 4. Mobile `style.css`

Responsive Layout:
- auf Handy volle Breite
- größere Touch-Zonen
- kein fixes 1100px-Spielfeld


In [ ]:
css = """* {
  box-sizing: border-box;
}

html, body {
  margin: 0;
  padding: 0;
  height: 100%;
  font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  background: #f3f4f6;
  color: #111827;
}

#app {
  min-height: 100vh;
  display: flex;
  flex-direction: column;
}

header {
  padding: 12px;
  background: #ffffff;
  border-bottom: 1px solid #d1d5db;
}

h1 {
  margin: 0 0 8px 0;
  font-size: 24px;
}

.controls {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
  align-items: center;
}

label {
  font-weight: 700;
}

select, button {
  font-size: 16px;
  padding: 10px 12px;
  border-radius: 10px;
  border: 1px solid #9ca3af;
  background: white;
}

button {
  background: #1f2937;
  color: white;
  border: none;
}

#status {
  margin-top: 10px;
  font-size: 16px;
  line-height: 1.35;
}

#message {
  margin-top: 8px;
  font-size: 16px;
  font-weight: 700;
}

main {
  flex: 1;
  padding: 8px;
  min-height: 0;
}

#board {
  position: relative;
  width: 100%;
  height: calc(100vh - 190px);
  min-height: 520px;
  background: #ffffff;
  border: 1px solid #d1d5db;
  border-radius: 14px;
  overflow: hidden;
  touch-action: manipulation;
}

.card {
  position: absolute;
  width: 180px;
  height: 80px;
  border: 3px solid #333;
  border-radius: 12px;
  text-align: center;
  padding-top: 20px;
  box-sizing: border-box;
  font-weight: 800;
  font-size: 22px;
  z-index: 2;
  color: white;
  text-shadow: 0 1px 2px rgba(0,0,0,0.45);
  user-select: none;
}

.zone {
  position: absolute;
  background: rgba(59, 130, 246, 0.18);
  border: 2px dashed rgba(37, 99, 235, 0.65);
  cursor: pointer;
  z-index: 1;
  border-radius: 8px;
}

.zone:hover,
.zone:active {
  background: rgba(59, 130, 246, 0.38);
}

@media (max-width: 700px) {
  h1 {
    font-size: 22px;
  }

  .controls {
    display: grid;
    grid-template-columns: 1fr 1fr;
  }

  .controls label {
    grid-column: 1 / -1;
  }

  .controls select {
    grid-column: 1 / -1;
    width: 100%;
  }

  button {
    width: 100%;
  }

  #board {
    height: calc(100vh - 230px);
    min-height: 430px;
  }
}
"""

(APP / "style.css").write_text(css, encoding="utf-8")
print("Created:", APP / "style.css")


## 5. `game.js`

Für den ersten Mobile-Schritt kopieren wir die aktuelle, funktionierende `game.js` aus deinem Browser-Prototyp.

Später können wir sie separat mobil optimieren, ohne das alte Notebook anzufassen.


In [ ]:
SOURCE_GAME = BASE / "game.js"
TARGET_GAME = APP / "game.js"

if SOURCE_GAME.exists():
    shutil.copyfile(SOURCE_GAME, TARGET_GAME)
    print("Copied:", SOURCE_GAME, "->", TARGET_GAME)
else:
    raise FileNotFoundError(f"Could not find {SOURCE_GAME}. Run this after your browser prototype has created game.js.")


## 6. PWA-Dateien: `manifest.json` und `service-worker.js`

Damit die App später auf dem Handy installiert und offline geladen werden kann.


In [ ]:
manifest = {
    "name": "Geo Poker",
    "short_name": "Geo Poker",
    "start_url": "./index.html",
    "display": "standalone",
    "background_color": "#f3f4f6",
    "theme_color": "#1f2937",
    "orientation": "portrait",
    "icons": [
        {
            "src": "icon-192.png",
            "sizes": "192x192",
            "type": "image/png"
        },
        {
            "src": "icon-512.png",
            "sizes": "512x512",
            "type": "image/png"
        }
    ]
}

(APP / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

service_worker = """const CACHE_NAME = "geo-poker-v1";

const FILES_TO_CACHE = [
  "./",
  "./index.html",
  "./style.css",
  "./game.js",
  "./places_data.js",
  "./country_outlines.js",
  "./manifest.json",
  "./icon-192.png",
  "./icon-512.png"
];

self.addEventListener("install", event => {
  event.waitUntil(
    caches.open(CACHE_NAME).then(cache => cache.addAll(FILES_TO_CACHE))
  );
});

self.addEventListener("fetch", event => {
  event.respondWith(
    caches.match(event.request).then(response => response || fetch(event.request))
  );
});
"""

(APP / "service-worker.js").write_text(service_worker, encoding="utf-8")

print("Created manifest and service worker.")


## 7. Einfache App-Icons erzeugen

Platzhalter-Icons. Die können wir später schöner machen.


In [ ]:
from PIL import Image, ImageDraw, ImageFont

def make_icon(path, size):
    img = Image.new("RGB", (size, size), "#1f2937")
    draw = ImageDraw.Draw(img)

    # simple compass-cross icon
    margin = size // 5
    center = size // 2
    draw.line((center, margin, center, size - margin), fill="#ffffff", width=size // 18)
    draw.line((margin, center, size - margin, center), fill="#ffffff", width=size // 18)

    r = size // 9
    colors = ["#e6194b", "#f58231", "#3cb44b", "#4363d8"]
    points = [
        (center, margin),
        (size - margin, center),
        (center, size - margin),
        (margin, center)
    ]

    for color, (x, y) in zip(colors, points):
        draw.ellipse((x-r, y-r, x+r, y+r), fill=color)

    img.save(path)

make_icon(APP / "icon-192.png", 192)
make_icon(APP / "icon-512.png", 512)

print("Created icons.")


## 8. Lokal testen

Für PWA/Service Worker brauchst du einen kleinen lokalen Webserver. Direktes Doppelklicken auf `index.html` reicht für PWA-Funktionen nicht.

In einem Terminal:

```bash
cd /Users/kathsch/Desktop/Uppsala/GeoPokerMobile
python3 -m http.server 8000
```

Dann im Browser öffnen:

```text
http://localhost:8000
```

Auf dem Handy muss der Ordner später irgendwo gehostet werden, z. B. GitHub Pages.
